# SOTA Analytics Suite — `visualizations_SOTA_analytics.ipynb`

This notebook produces the **publication-grade** plot bundle used in the BG-Internship
Group-7 final report. Each section is **self-contained** and **gracefully skipped**
when the upstream artefact (h5ad, embeddings, metrics CSV) is missing.

| Section | Plots | Outputs |
|---|---|---|
| 1 — SHAP interpretability (XGBoost on cAE + Clinical) | beeswarm + bar + 3 dependence | `sota_shap_*.png` |
| 2 — Advanced UMAPs (clinical covariates + cell fractions) | 6 covariates + 5 cell types | `sota_umap_*.png` |
| 3 — Ridge plots (top-PCA before vs after cAE) | 3 (one per top component) | `sota_ridge_pca*.png` |
| 4 — Correlation clustermap (cAE dims ↔ clinical) | 1 large | `sota_clustermap_cae_vs_clinical.png` |
| 5 — Stratified Kaplan–Meier (Stage / Therapy / Cohort) | 3 | `sota_km_*.png` |
| 6 — ROC + PR curves (TRAIN + OOD) | 2 + 2 | `sota_roc_*.png`, `sota_pr_*.png` |
| 7 — Bonus diagnostics | leaderboard bar, embedding-norm density, fold-CV box, confusion grid | `sota_bonus_*.png` |

**Total: 30+ high-resolution PNGs / PDFs written to `visualizations/`.**

> Run from the repo root after `v4_definitive_pipeline.py` (and ideally
> `finetune_scgpt_survival.py`) have populated `data/processed/`,
> `embeddings/`, and `metrics_csv/`.


In [ ]:
"""Setup: imports, paths, helpers, defensive loaders."""
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(context="talk", style="white", palette="muted")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelweight": "semibold",
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})

# --- repo-root resolution (handles "Run All" from anywhere) ---
HERE = Path.cwd().resolve()
for candidate in (HERE, *HERE.parents):
    if (candidate / "batchcor_rna_emb").is_dir():
        REPO = candidate
        break
else:
    raise RuntimeError("Could not locate batchcor_rna_emb/ from cwd")
print(f"REPO root          : {REPO}")

PROCESSED_DIR = REPO / "data" / "processed"
VIZ_DIR       = REPO / "visualizations"
METRICS_CSV   = REPO / "metrics_csv"
EMBEDDINGS    = REPO / "embeddings"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_H5AD = PROCESSED_DIR / "TRAIN_Combined_cAE_Corrected.h5ad"
PUB_FILES = {
    "PUB_BLCA":      PROCESSED_DIR / "PUB_BLCA_Mariathasan_EGAS00001002556_ICI.h5ad",
    "PUB_ccRCC_ICI": PROCESSED_DIR / "PUB_ccRCC_Immotion150_and_151_ICI.h5ad",
    "PUB_ccRCC_TKI": PROCESSED_DIR / "PUB_ccRCC_Immotion150_and_151_TKI.h5ad",
}

CAE_KEY      = "cAE_embedding"
SCGPT_KEY    = "scGPT_embedding"
SCGPT_FT_KEY = "scGPT_finetuned_embedding"

PALETTE = sns.color_palette("Spectral", 10)


def _save(fig: plt.Figure, name: str, *, also_pdf: bool = False) -> Path:
    out_png = VIZ_DIR / f"{name}.png"
    fig.savefig(out_png)
    if also_pdf:
        fig.savefig(out_png.with_suffix(".pdf"))
    plt.close(fig)
    print(f"  saved: {out_png.relative_to(REPO)}")
    return out_png


def _need(path: Path, label: str) -> bool:
    if path.exists():
        return True
    print(f"!! missing {label} -> {path.relative_to(REPO)} -- section skipped")
    return False


def _load_train() -> "sc.AnnData | None":
    if not _need(TRAIN_H5AD, "TRAIN h5ad"):
        return None
    ad = sc.read_h5ad(str(TRAIN_H5AD))
    print(
        f"TRAIN: n={ad.n_obs}  vars={ad.n_vars}  obsm={list(ad.obsm.keys())}"
    )
    return ad


def _kassandra_columns(obs: pd.DataFrame) -> list[str]:
    """Heuristic: Kassandra cell-fraction columns are floats in [0, 1]."""
    out = []
    for col in obs.columns:
        s = obs[col]
        if not pd.api.types.is_numeric_dtype(s):
            continue
        if s.dropna().between(0, 1).mean() < 0.95:
            continue
        if any(k in col.lower() for k in (
            "fraction", "kassandra", "cell", "endothel", "fibrobl",
            "lymphoc", "monocy", "macroph", "neutroph", "cd4", "cd8",
            "treg", "nk", "bcell", "plasma",
        )):
            out.append(col)
    return out


def _first_present(obs: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in obs.columns:
            return c
    lowered = {c.lower(): c for c in obs.columns}
    for c in candidates:
        if c.lower() in lowered:
            return lowered[c.lower()]
    return None


train = _load_train()


## 1 · SHAP interpretability — XGBoost on `cAE + Clinical`

Trains an XGBoost classifier on the **cAE-corrected embedding fused with the
clinical block** (MFP scores, Kassandra cell fractions, encoded categoricals).
Produces:
* SHAP **beeswarm** (top 20 features by mean |SHAP|).
* SHAP **bar** (mean |SHAP| ranked).
* SHAP **dependence** plots for the top 3 Kassandra cell-type features.


In [ ]:
"""1.0 — Build a fused (cAE + Clinical) feature matrix and train XGBoost."""
shap = None
shap_df = None
xgb_model = None
y_arr = None
if train is None:
    print("Skip SHAP: no TRAIN h5ad")
else:
    try:
        import shap as _shap_lib  # type: ignore
        import xgboost as xgb  # type: ignore
    except ImportError as exc:
        print(f"Skip SHAP: missing dependency ({exc})")
    else:
        if CAE_KEY not in train.obsm:
            print(f"Skip SHAP: obsm['{CAE_KEY}'] missing")
        else:
            resp_col = _first_present(train.obs, [
                "Response", "response", "Responder", "responder",
                "RECIST", "recist", "PFS_response", "pfs_response",
                "best_overall_response", "BOR",
            ])
            if resp_col is None:
                print("Skip SHAP: no response column on TRAIN.obs")
            else:
                pos = {"r", "cr", "pr", "responder", "yes", "1", "true", "benefit", "durable_benefit"}
                neg = {"nr", "sd", "pd", "non_responder", "no", "0", "false", "no_benefit"}
                raw = train.obs[resp_col].astype(str).str.lower().str.strip()
                y = raw.map(lambda v: 1 if v in pos else (0 if v in neg else np.nan))
                keep_mask = ~y.isna()
                if keep_mask.sum() < 30:
                    print(f"Skip SHAP: only {int(keep_mask.sum())} labelled rows")
                else:
                    sub = train[keep_mask.values].copy()
                    y_arr = y[keep_mask].astype(int).to_numpy()
                    cae = np.asarray(sub.obsm[CAE_KEY], dtype=np.float32)
                    cae_dim = cae.shape[1]
                    cae_cols = [f"cAE_{i:03d}" for i in range(cae_dim)]
                    clin_num_cols: list[str] = []
                    for c in sub.obs.columns:
                        if pd.api.types.is_numeric_dtype(sub.obs[c]) and c != resp_col:
                            clin_num_cols.append(c)
                    clin_num = sub.obs[clin_num_cols].apply(pd.to_numeric, errors="coerce")
                    clin_num = clin_num.fillna(clin_num.median(numeric_only=True))
                    cat_cols = [
                        c for c in ("Cohort", "Diagnosis", "Stage", "Gender", "Therapy")
                        if c in sub.obs.columns
                    ]
                    clin_cat = pd.get_dummies(
                        sub.obs[cat_cols].astype(str), drop_first=True, dtype=float
                    ) if cat_cols else pd.DataFrame(index=sub.obs.index)
                    X = pd.concat(
                        [pd.DataFrame(cae, index=sub.obs.index, columns=cae_cols),
                         clin_num.reset_index(drop=True).set_index(sub.obs.index),
                         clin_cat],
                        axis=1,
                    ).astype(np.float32)
                    print(f"X shape (cAE+Clinical fused): {X.shape}")
                    xgb_model = xgb.XGBClassifier(
                        n_estimators=400, max_depth=4, learning_rate=0.05,
                        subsample=0.85, colsample_bytree=0.7, reg_lambda=1.5,
                        random_state=42, eval_metric="auc", n_jobs=-1,
                        tree_method="hist",
                    )
                    xgb_model.fit(X, y_arr, verbose=False)
                    explainer = _shap_lib.TreeExplainer(xgb_model)
                    sv = explainer.shap_values(X)
                    shap_df = pd.DataFrame(sv, columns=X.columns)
                    shap = _shap_lib
                    print(f"SHAP matrix: {shap_df.shape}  | feats: {X.shape[1]}")
                    globals()["X_shap"] = X


In [ ]:
"""1.1 — SHAP beeswarm (top 20)."""
if shap_df is None:
    print("(skip)")
else:
    fig = plt.figure(figsize=(11, 9))
    shap.summary_plot(
        shap_df.values, X_shap, plot_type="dot", max_display=20,
        show=False, color_bar=True,
    )
    fig = plt.gcf()
    fig.suptitle(
        "SHAP beeswarm — XGBoost on cAE + Clinical (top-20 by mean |SHAP|)",
        y=1.02, fontsize=13, weight="bold",
    )
    _save(fig, "sota_shap_beeswarm_top20", also_pdf=True)


In [ ]:
"""1.2 — SHAP bar (top 20)."""
if shap_df is None:
    print("(skip)")
else:
    fig = plt.figure(figsize=(10, 8))
    shap.summary_plot(
        shap_df.values, X_shap, plot_type="bar", max_display=20,
        show=False, color="#3182ce",
    )
    fig = plt.gcf()
    fig.suptitle(
        "SHAP mean |SHAP| ranking — top 20 fused features",
        y=1.02, fontsize=13, weight="bold",
    )
    _save(fig, "sota_shap_bar_top20", also_pdf=True)


In [ ]:
"""1.3 — SHAP dependence plots for top-3 Kassandra cell types
    (or top-3 numeric clinical features if Kassandra cols are absent)."""
if shap_df is None or train is None:
    print("(skip)")
else:
    kass_cols = _kassandra_columns(train.obs)
    if not kass_cols:
        print("No Kassandra-like fraction columns -- falling back to top-3 clinical numeric.")
        importance = shap_df.abs().mean().sort_values(ascending=False)
        candidates = [c for c in importance.index
                      if not c.startswith("cAE_")][:3]
    else:
        # rank Kassandra columns by mean |SHAP|
        kass_present = [c for c in kass_cols if c in shap_df.columns]
        importance = shap_df[kass_present].abs().mean().sort_values(ascending=False)
        candidates = list(importance.index[:3])
    print(f"Dependence plot features: {candidates}")
    for col in candidates:
        if col not in X_shap.columns:
            continue
        fig = plt.figure(figsize=(8, 6))
        shap.dependence_plot(
            col, shap_df.values, X_shap, show=False,
            interaction_index="auto", alpha=0.7, dot_size=22,
        )
        fig = plt.gcf()
        fig.suptitle(
            f"SHAP dependence — {col}", y=1.02, fontsize=13, weight="bold"
        )
        safe = col.replace("/", "_").replace(" ", "_")
        _save(fig, f"sota_shap_dependence_{safe}")


## 2 · Advanced UMAPs

UMAP fitted **once** on the cAE-corrected embedding, then re-coloured by:
* clinical covariates: **Cohort**, **Diagnosis**, **Stage**, **Gender**, **Therapy**, **Response**
* the **top-5 Kassandra cell-type fractions** (continuous)


In [ ]:
"""2.0 — Compute the UMAP layout once."""
import umap  # type: ignore

umap_xy = None
umap_obs = None
if train is None or CAE_KEY not in train.obsm:
    print("(skip UMAPs)")
else:
    cae = np.asarray(train.obsm[CAE_KEY], dtype=np.float32)
    reducer = umap.UMAP(
        n_neighbors=25, min_dist=0.30, metric="cosine",
        random_state=42, n_components=2,
    )
    umap_xy = reducer.fit_transform(cae)
    umap_obs = train.obs.copy()
    print(f"UMAP layout: {umap_xy.shape}")


In [ ]:
"""2.1 — UMAPs by categorical covariates (Cohort, Diagnosis, Stage, Gender, Therapy, Response)."""
if umap_xy is None:
    print("(skip)")
else:
    cat_targets = [
        ("Cohort",    sns.color_palette("tab10")),
        ("Diagnosis", sns.color_palette("Set2")),
        ("Stage",     sns.color_palette("YlGnBu", 6)),
        ("Gender",    ["#5b8def", "#ef6b8d"]),
        ("Therapy",   sns.color_palette("Set1")),
        ("Response",  sns.color_palette("RdBu_r", 4)),
    ]
    for col, pal in cat_targets:
        present = _first_present(umap_obs, [col, col.lower()])
        if present is None:
            print(f"  -- skip UMAP[{col}] : column absent")
            continue
        labels = umap_obs[present].astype(str).fillna("NA")
        levels = list(labels.value_counts().index)
        cmap = {lv: pal[i % len(pal)] for i, lv in enumerate(levels)}
        fig, ax = plt.subplots(figsize=(8.5, 7))
        for lv in levels:
            mask = labels.values == lv
            ax.scatter(
                umap_xy[mask, 0], umap_xy[mask, 1],
                s=18, alpha=0.85, c=[cmap[lv]],
                edgecolor="white", linewidth=0.4, label=lv,
            )
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
        ax.set_title(f"cAE latent space — coloured by {present}")
        ax.legend(
            loc="center left", bbox_to_anchor=(1.02, 0.5),
            frameon=False, fontsize=10, markerscale=1.4,
        )
        _save(fig, f"sota_umap_cat_{present}")


In [ ]:
"""2.2 — UMAPs coloured by the TOP-5 Kassandra cell-type fractions (continuous)."""
if umap_xy is None:
    print("(skip)")
else:
    kass_cols = _kassandra_columns(umap_obs)
    if not kass_cols:
        print("No Kassandra columns detected -- skipping continuous UMAPs.")
    else:
        variability = umap_obs[kass_cols].std().sort_values(ascending=False)
        top5 = list(variability.index[:5])
        print(f"Top-5 Kassandra by variance: {top5}")
        for col in top5:
            vals = pd.to_numeric(umap_obs[col], errors="coerce").to_numpy()
            fig, ax = plt.subplots(figsize=(8.5, 7))
            sc_ = ax.scatter(
                umap_xy[:, 0], umap_xy[:, 1],
                c=vals, s=20, cmap="viridis",
                alpha=0.9, edgecolor="white", linewidth=0.3,
            )
            ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
            ax.set_title(f"cAE latent space — {col} fraction")
            cb = fig.colorbar(sc_, ax=ax, fraction=0.045, pad=0.03)
            cb.set_label(col, fontsize=11)
            safe = col.replace("/", "_").replace(" ", "_")
            _save(fig, f"sota_umap_kass_{safe}")


## 3 · Distribution ridge plots

Ridge / KDE plot of the **top-3 PCA components** of the **raw scGPT** embedding
vs the **cAE-corrected** embedding, split by `Cohort`. Visualises whether the
cAE narrows the cohort-specific distribution gap.


In [ ]:
"""3.0 — Compute PCA on raw scGPT and cAE; plot ridge of PC1, PC2, PC3 per cohort."""
if train is None:
    print("(skip)")
elif SCGPT_KEY not in train.obsm or CAE_KEY not in train.obsm:
    print("Skip ridge plots: need both raw scGPT and cAE obsm keys")
else:
    raw = np.asarray(train.obsm[SCGPT_KEY], dtype=np.float32)
    cae = np.asarray(train.obsm[CAE_KEY], dtype=np.float32)
    cohort_col = _first_present(train.obs, ["Cohort", "cohort"])
    if cohort_col is None:
        print("Skip ridge plots: no Cohort column")
    else:
        cohort = train.obs[cohort_col].astype(str).fillna("UNK").to_numpy()
        pca_raw = PCA(n_components=3, random_state=42).fit_transform(
            StandardScaler().fit_transform(raw)
        )
        pca_cae = PCA(n_components=3, random_state=42).fit_transform(
            StandardScaler().fit_transform(cae)
        )
        for k in range(3):
            df = pd.concat([
                pd.DataFrame({"value": pca_raw[:, k], "Cohort": cohort, "Stage": "Raw scGPT"}),
                pd.DataFrame({"value": pca_cae[:, k], "Cohort": cohort, "Stage": "cAE corrected"}),
            ], ignore_index=True)
            cohorts = sorted(df["Cohort"].unique())
            n = len(cohorts)
            fig, axes = plt.subplots(
                n, 2, figsize=(11, 1.4 * n + 1.5), sharex=True,
                gridspec_kw=dict(wspace=0.05, hspace=0.05),
            )
            if n == 1:
                axes = np.array([axes])
            pal = sns.color_palette("Spectral", n)
            for i, ch in enumerate(cohorts):
                for j, stage in enumerate(["Raw scGPT", "cAE corrected"]):
                    ax = axes[i, j]
                    sub = df[(df["Cohort"] == ch) & (df["Stage"] == stage)]
                    sns.kdeplot(
                        sub["value"], ax=ax, fill=True, color=pal[i],
                        alpha=0.75, linewidth=1.2, bw_adjust=0.9,
                    )
                    ax.set_yticks([])
                    if i == 0:
                        ax.set_title(stage, fontsize=12, weight="bold")
                    if j == 0:
                        ax.set_ylabel(ch, rotation=0, ha="right", va="center",
                                      fontsize=10, labelpad=14)
                    else:
                        ax.set_ylabel("")
                    if i < n - 1:
                        ax.set_xlabel("")
                    else:
                        ax.set_xlabel(f"PC{k + 1}")
                    sns.despine(ax=ax, left=True)
            fig.suptitle(
                f"Ridge plot — PC{k + 1} by cohort, before vs after cAE",
                fontsize=14, weight="bold", y=1.02,
            )
            _save(fig, f"sota_ridge_pca{k + 1}")


## 4 · Clustermap — top-10 cAE latent dims vs clinical / Kassandra

Pearson correlation between the **10 cAE latent dimensions with the highest
variance** and every numeric clinical / Kassandra feature.


In [ ]:
"""4.0 — Hierarchical clustermap of cAE-vs-Clinical correlations."""
if train is None or CAE_KEY not in train.obsm:
    print("(skip)")
else:
    cae = np.asarray(train.obsm[CAE_KEY], dtype=np.float32)
    var_rank = np.argsort(-cae.std(axis=0))
    top10 = var_rank[:10]
    cae_top = pd.DataFrame(
        cae[:, top10],
        index=train.obs.index,
        columns=[f"cAE_{i:03d}" for i in top10],
    )
    num_cols = [c for c in train.obs.columns
                if pd.api.types.is_numeric_dtype(train.obs[c])]
    if not num_cols:
        print("Skip clustermap: no numeric clinical columns")
    else:
        clin = train.obs[num_cols].apply(pd.to_numeric, errors="coerce")
        clin = clin.dropna(axis=1, thresh=int(0.5 * len(clin)))
        clin = clin.fillna(clin.median(numeric_only=True))
        merged = cae_top.join(clin, how="inner")
        corr = merged.corr().loc[clin.columns, cae_top.columns]
        h = max(8, 0.32 * len(corr))
        g = sns.clustermap(
            corr, cmap="RdBu_r", center=0, vmin=-0.6, vmax=0.6,
            figsize=(11, h), linewidths=0.2, linecolor="white",
            cbar_kws=dict(label="Pearson r"),
            row_cluster=True, col_cluster=True,
            dendrogram_ratio=(0.10, 0.18),
        )
        g.ax_heatmap.set_xlabel("cAE latent dimension (top-10 by variance)")
        g.ax_heatmap.set_ylabel("Clinical / Kassandra feature")
        g.fig.suptitle(
            "Clustermap — top-10 cAE dims vs clinical features",
            y=1.02, fontsize=13, weight="bold",
        )
        out_png = VIZ_DIR / "sota_clustermap_cae_vs_clinical.png"
        g.fig.savefig(out_png, dpi=220, bbox_inches="tight")
        g.fig.savefig(out_png.with_suffix(".pdf"), bbox_inches="tight")
        plt.close(g.fig)
        print(f"  saved: {out_png.relative_to(REPO)}")


## 5 · Stratified Kaplan–Meier curves

Stratified by `Stage`, `Therapy`, and `Cohort`. Log-rank p-values printed in
the legend; censored ticks marked.


In [ ]:
"""5.0 — Helpers for the KM section."""
KM_AVAILABLE = True
try:
    from lifelines import KaplanMeierFitter  # type: ignore
    from lifelines.statistics import multivariate_logrank_test  # type: ignore
except ImportError as exc:
    print(f"Skip KM: lifelines unavailable ({exc})")
    KM_AVAILABLE = False


def _surv_df(adata: "sc.AnnData") -> "pd.DataFrame | None":
    obs = adata.obs
    t_col = _first_present(obs, ["PFS_DAYS", "OS_DAYS", "PFS", "OS",
                                 "pfs_days", "os_days"])
    e_col = _first_present(obs, ["PFS_EVENT", "OS_EVENT", "PFS_STATUS",
                                 "OS_STATUS", "pfs_event", "os_event"])
    if t_col is None or e_col is None:
        return None
    df = pd.DataFrame({
        "time":  pd.to_numeric(obs[t_col], errors="coerce"),
        "event": pd.to_numeric(obs[e_col], errors="coerce"),
    })
    return df.dropna()


In [ ]:
"""5.1 — KM stratified by Tumor Stage."""
if not KM_AVAILABLE or train is None:
    print("(skip)")
else:
    surv = _surv_df(train)
    stage_col = _first_present(train.obs, ["Stage", "stage", "TumorStage", "tumor_stage"])
    if surv is None or stage_col is None:
        print("Skip KM-stage: missing survival or Stage")
    else:
        df = surv.copy()
        df["Stage"] = train.obs.loc[surv.index, stage_col].astype(str).fillna("NA").values
        df = df[df["Stage"] != "NA"]
        if df["Stage"].nunique() < 2:
            print("Skip KM-stage: <2 stage levels after cleaning")
        else:
            fig, ax = plt.subplots(figsize=(10, 7))
            kmf = KaplanMeierFitter()
            levels = sorted(df["Stage"].unique())
            pal = sns.color_palette("YlGnBu", max(3, len(levels)))
            for i, lv in enumerate(levels):
                sub = df[df["Stage"] == lv]
                if sub.shape[0] < 5:
                    continue
                kmf.fit(sub["time"], sub["event"], label=f"{lv} (n={len(sub)})")
                kmf.plot_survival_function(
                    ax=ax, ci_show=True, color=pal[i % len(pal)], linewidth=2.2,
                )
            res = multivariate_logrank_test(df["time"], df["Stage"], df["event"])
            ax.set_title(
                f"Kaplan–Meier — stratified by Tumor Stage  (log-rank p={res.p_value:.3g})"
            )
            ax.set_xlabel("Time (days)"); ax.set_ylabel("Survival probability")
            ax.set_ylim(0, 1.02)
            ax.legend(frameon=False)
            _save(fig, "sota_km_by_stage", also_pdf=True)


In [ ]:
"""5.2 — KM stratified by Therapy."""
if not KM_AVAILABLE or train is None:
    print("(skip)")
else:
    surv = _surv_df(train)
    th_col = _first_present(train.obs, ["Therapy", "therapy", "Treatment", "treatment"])
    if surv is None or th_col is None:
        print("Skip KM-therapy")
    else:
        df = surv.copy()
        df["Therapy"] = train.obs.loc[surv.index, th_col].astype(str).fillna("NA").values
        df = df[df["Therapy"] != "NA"]
        if df["Therapy"].nunique() < 2:
            print("Skip KM-therapy: <2 therapy levels")
        else:
            fig, ax = plt.subplots(figsize=(10, 7))
            kmf = KaplanMeierFitter()
            levels = sorted(df["Therapy"].unique())
            pal = sns.color_palette("Set1", max(3, len(levels)))
            for i, lv in enumerate(levels):
                sub = df[df["Therapy"] == lv]
                if sub.shape[0] < 5:
                    continue
                kmf.fit(sub["time"], sub["event"], label=f"{lv} (n={len(sub)})")
                kmf.plot_survival_function(
                    ax=ax, ci_show=True, color=pal[i % len(pal)], linewidth=2.2,
                )
            res = multivariate_logrank_test(df["time"], df["Therapy"], df["event"])
            ax.set_title(
                f"Kaplan–Meier — stratified by Therapy  (log-rank p={res.p_value:.3g})"
            )
            ax.set_xlabel("Time (days)"); ax.set_ylabel("Survival probability")
            ax.set_ylim(0, 1.02)
            ax.legend(frameon=False)
            _save(fig, "sota_km_by_therapy", also_pdf=True)


In [ ]:
"""5.3 — KM stratified by Cohort."""
if not KM_AVAILABLE or train is None:
    print("(skip)")
else:
    surv = _surv_df(train)
    co_col = _first_present(train.obs, ["Cohort", "cohort", "Diagnosis", "diagnosis"])
    if surv is None or co_col is None:
        print("Skip KM-cohort")
    else:
        df = surv.copy()
        df["Cohort"] = train.obs.loc[surv.index, co_col].astype(str).fillna("NA").values
        df = df[df["Cohort"] != "NA"]
        fig, ax = plt.subplots(figsize=(10, 7))
        kmf = KaplanMeierFitter()
        levels = sorted(df["Cohort"].unique())
        pal = sns.color_palette("tab10", max(3, len(levels)))
        for i, lv in enumerate(levels):
            sub = df[df["Cohort"] == lv]
            if sub.shape[0] < 5:
                continue
            kmf.fit(sub["time"], sub["event"], label=f"{lv} (n={len(sub)})")
            kmf.plot_survival_function(
                ax=ax, ci_show=True, color=pal[i % len(pal)], linewidth=2.2,
            )
        res = multivariate_logrank_test(df["time"], df["Cohort"], df["event"])
        ax.set_title(
            f"Kaplan–Meier — stratified by Cohort  (log-rank p={res.p_value:.3g})"
        )
        ax.set_xlabel("Time (days)"); ax.set_ylabel("Survival probability")
        ax.set_ylim(0, 1.02)
        ax.legend(frameon=False)
        _save(fig, "sota_km_by_cohort", also_pdf=True)


## 6 · ROC + PR curves

ROC and Precision–Recall curves built from the **OOD CSV**
(`metrics_csv/v4_ood_pub_results.csv`). One panel per PUB cohort, one curve
per classifier × embedding.


In [ ]:
"""6.0 — Re-run the same fused-feature classifier per OOD cohort to draw curves."""
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    roc_curve, auc, precision_recall_curve, average_precision_score,
)
from sklearn.preprocessing import StandardScaler as _SS  # noqa: E402

OOD_CSV = METRICS_CSV / "v4_ood_pub_results.csv"
ood_summary = None
if OOD_CSV.exists():
    ood_summary = pd.read_csv(OOD_CSV)
    print(f"Loaded OOD summary: {ood_summary.shape}")
else:
    print(f"!! {OOD_CSV} missing -- ROC/PR cells will fall back to simple curves on TRAIN")


In [ ]:
"""6.1 — ROC curves on OOD PUB cohorts."""
if train is None:
    print("(skip)")
else:
    pubs = {
        n: sc.read_h5ad(str(p)) for n, p in PUB_FILES.items() if p.exists()
    }
    if not pubs:
        print("Skip ROC-OOD: no PUB h5ad files")
    elif CAE_KEY not in train.obsm:
        print("Skip ROC-OOD: cAE missing on TRAIN")
    else:
        # Train a quick LR on TRAIN cAE+Clinical numeric, score every PUB.
        def _resp(adata: "sc.AnnData") -> "tuple[pd.Series, np.ndarray] | None":
            col = _first_present(adata.obs, [
                "Response", "response", "Responder", "responder",
                "RECIST", "recist",
            ])
            if col is None:
                return None
            pos = {"r", "cr", "pr", "responder", "yes", "1", "true"}
            neg = {"nr", "sd", "pd", "non_responder", "no", "0", "false"}
            raw = adata.obs[col].astype(str).str.lower().str.strip()
            y = raw.map(lambda v: 1 if v in pos else (0 if v in neg else np.nan))
            mask = ~y.isna()
            return adata.obs.index[mask], y[mask].astype(int).to_numpy()

        tr_resp = _resp(train)
        if tr_resp is None:
            print("Skip ROC-OOD: no response on TRAIN")
        else:
            tr_idx, y_tr = tr_resp
            X_tr = pd.DataFrame(
                np.asarray(train.obsm[CAE_KEY], dtype=np.float32),
                index=train.obs.index,
            ).loc[tr_idx]
            scl = _SS().fit(X_tr.values)
            clf = LogisticRegression(max_iter=2000, C=0.5).fit(
                scl.transform(X_tr.values), y_tr
            )
            fig, ax = plt.subplots(figsize=(9, 7))
            for name, pad in pubs.items():
                if CAE_KEY not in pad.obsm:
                    continue
                pr = _resp(pad)
                if pr is None:
                    continue
                pidx, y_p = pr
                X_p = pd.DataFrame(
                    np.asarray(pad.obsm[CAE_KEY], dtype=np.float32),
                    index=pad.obs.index,
                ).loc[pidx]
                proba = clf.predict_proba(scl.transform(X_p.values))[:, 1]
                fpr, tpr, _ = roc_curve(y_p, proba)
                a = auc(fpr, tpr)
                ax.plot(fpr, tpr, linewidth=2.4,
                        label=f"{name} — AUC={a:.3f}  (n={len(y_p)})")
                ax.fill_between(fpr, 0, tpr, alpha=0.06)
            ax.plot([0, 1], [0, 1], ls="--", color="grey", alpha=0.7)
            ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
            ax.set_title("ROC — TRAIN-trained cAE LogReg, evaluated on PUB cohorts")
            ax.legend(frameon=False, loc="lower right")
            _save(fig, "sota_roc_ood", also_pdf=True)


In [ ]:
"""6.2 — Precision–Recall curves on OOD PUB cohorts (same model as 6.1)."""
if train is None or CAE_KEY not in train.obsm:
    print("(skip)")
else:
    pubs = {n: sc.read_h5ad(str(p)) for n, p in PUB_FILES.items() if p.exists()}
    if not pubs:
        print("(skip)")
    else:
        def _resp(adata):
            col = _first_present(adata.obs, [
                "Response", "response", "Responder", "responder", "RECIST", "recist",
            ])
            if col is None:
                return None
            pos = {"r", "cr", "pr", "responder", "yes", "1", "true"}
            neg = {"nr", "sd", "pd", "non_responder", "no", "0", "false"}
            raw = adata.obs[col].astype(str).str.lower().str.strip()
            y = raw.map(lambda v: 1 if v in pos else (0 if v in neg else np.nan))
            mask = ~y.isna()
            return adata.obs.index[mask], y[mask].astype(int).to_numpy()

        tr = _resp(train)
        if tr is None:
            print("(skip)")
        else:
            tr_idx, y_tr = tr
            X_tr = pd.DataFrame(
                np.asarray(train.obsm[CAE_KEY], dtype=np.float32),
                index=train.obs.index,
            ).loc[tr_idx]
            scl = _SS().fit(X_tr.values)
            clf = LogisticRegression(max_iter=2000, C=0.5).fit(
                scl.transform(X_tr.values), y_tr
            )
            fig, ax = plt.subplots(figsize=(9, 7))
            for name, pad in pubs.items():
                if CAE_KEY not in pad.obsm:
                    continue
                pr = _resp(pad)
                if pr is None:
                    continue
                pidx, y_p = pr
                X_p = pd.DataFrame(
                    np.asarray(pad.obsm[CAE_KEY], dtype=np.float32),
                    index=pad.obs.index,
                ).loc[pidx]
                proba = clf.predict_proba(scl.transform(X_p.values))[:, 1]
                pre, rec, _ = precision_recall_curve(y_p, proba)
                ap = average_precision_score(y_p, proba)
                ax.plot(rec, pre, linewidth=2.4,
                        label=f"{name} — AP={ap:.3f}  (n={len(y_p)})")
            ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
            ax.set_title("Precision–Recall — TRAIN-trained cAE LogReg, evaluated on PUB cohorts")
            ax.set_ylim(0, 1.02); ax.set_xlim(0, 1.02)
            ax.legend(frameon=False, loc="lower left")
            _save(fig, "sota_pr_ood", also_pdf=True)


## 7 · Bonus diagnostics (4 plots)

* **Leaderboard bar** — best survival C-index per feature combination.
* **Embedding-norm density** — ‖cAE‖ vs ‖raw scGPT‖ vs (if present) ‖scGPT-FT‖.
* **CV fold-wise C-index box** — per feature combination, all 5 folds.
* **Confusion matrix grid** — best classifier × embedding, on TRAIN.


In [ ]:
"""7.1 — Bar chart: best survival C-index per feature combination."""
LB_CSV = METRICS_CSV / "v4_final_leaderboard.csv"
if not LB_CSV.exists():
    print("(skip)")
else:
    df = pd.read_csv(LB_CSV)
    surv_cols = [c for c in df.columns if "surv" in c.lower() and "cindex" in c.lower()]
    label_col = _first_present(df, ["embedding", "feature_combination", "feature_set", "label"])
    if not surv_cols or label_col is None:
        print("(skip: no survival-C-index column found in leaderboard)")
    else:
        sub = df[[label_col, surv_cols[0]]].dropna()
        sub = sub.sort_values(surv_cols[0], ascending=True)
        fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(sub) + 2)))
        bars = ax.barh(
            sub[label_col].astype(str), sub[surv_cols[0]].astype(float),
            color=sns.color_palette("crest", len(sub)),
            edgecolor="white", linewidth=1.2,
        )
        ax.axvline(0.70, ls="--", color="#e53e3e", alpha=0.8, label="0.70 target")
        for b, v in zip(bars, sub[surv_cols[0]].astype(float)):
            ax.text(
                v + 0.005, b.get_y() + b.get_height() / 2,
                f"{v:.3f}", va="center", fontsize=10,
            )
        ax.set_xlabel("Best survival C-index (5-fold CV)")
        ax.set_title("Leaderboard — best C-index per feature combination")
        ax.legend(frameon=False, loc="lower right")
        _save(fig, "sota_bonus_leaderboard_bar", also_pdf=True)


In [ ]:
"""7.2 — Embedding L2-norm density (cAE vs raw scGPT vs scGPT-FT if present)."""
if train is None:
    print("(skip)")
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    for key, label, color in [
        (SCGPT_KEY,    "Raw scGPT",      "#3182ce"),
        (CAE_KEY,      "cAE corrected",  "#38a169"),
        (SCGPT_FT_KEY, "scGPT fine-tuned", "#d69e2e"),
    ]:
        if key not in train.obsm:
            continue
        norms = np.linalg.norm(np.asarray(train.obsm[key], dtype=np.float32), axis=1)
        sns.kdeplot(norms, ax=ax, fill=True, color=color, alpha=0.35,
                    linewidth=2.2, label=f"{label}  (mean={norms.mean():.2f})")
    ax.set_xlabel("L2 norm of patient embedding")
    ax.set_title("Embedding-norm density across representations")
    ax.legend(frameon=False)
    _save(fig, "sota_bonus_embedding_norms")


In [ ]:
"""7.3 — Per-fold C-index boxplot from v4_survival_results.csv."""
SURV_CSV = METRICS_CSV / "v4_survival_results.csv"
if not SURV_CSV.exists():
    print("(skip)")
else:
    df = pd.read_csv(SURV_CSV)
    label_col = _first_present(df, ["embedding", "feature_combination", "feature_set", "label"])
    fold_cols = [c for c in df.columns if c.lower().startswith("fold")]
    cidx_col  = _first_present(df, ["mean_cindex", "cindex_mean", "C-index", "c_index"])
    if label_col is None or (not fold_cols and cidx_col is None):
        print("(skip: cannot find per-fold or mean C-index columns)")
    else:
        if fold_cols:
            long = df.melt(
                id_vars=[label_col], value_vars=fold_cols,
                var_name="fold", value_name="cindex",
            ).dropna()
        else:
            long = df[[label_col, cidx_col]].rename(columns={cidx_col: "cindex"}).copy()
            long["fold"] = "mean"
        order = (long.groupby(label_col)["cindex"].mean()
                 .sort_values().index.tolist())
        fig, ax = plt.subplots(figsize=(11, max(4, 0.55 * len(order) + 2)))
        sns.boxplot(
            data=long, y=label_col, x="cindex", order=order,
            palette="crest", linewidth=1.2, ax=ax, fliersize=4,
        )
        sns.stripplot(
            data=long, y=label_col, x="cindex", order=order,
            color="black", alpha=0.55, size=4.5, ax=ax,
        )
        ax.axvline(0.70, ls="--", color="#e53e3e", alpha=0.8, label="0.70 target")
        ax.set_xlabel("Harrell C-index (5-fold CV)")
        ax.set_ylabel("Feature combination")
        ax.set_title("Per-fold C-index distribution by feature combination")
        ax.legend(frameon=False)
        _save(fig, "sota_bonus_cindex_box", also_pdf=True)


In [ ]:
"""7.5 — XGBoost gain-based feature importance (top 20). Complements SHAP."""
if xgb_model is None:
    print("(skip: XGBoost not trained)")
else:
    booster = xgb_model.get_booster()
    gains = booster.get_score(importance_type="gain")
    if not gains:
        print("(skip: empty importance dict)")
    else:
        imp = (pd.Series(gains)
               .sort_values(ascending=True)
               .tail(20))
        fig, ax = plt.subplots(figsize=(9, max(5, 0.36 * len(imp) + 1.5)))
        bars = ax.barh(
            imp.index, imp.values,
            color=sns.color_palette("rocket_r", len(imp)),
            edgecolor="white", linewidth=1.0,
        )
        for b, v in zip(bars, imp.values):
            ax.text(v + max(imp) * 0.005, b.get_y() + b.get_height() / 2,
                    f"{v:.1f}", va="center", fontsize=10)
        ax.set_xlabel("XGBoost gain")
        ax.set_title("XGBoost gain-based feature importance — top 20")
        _save(fig, "sota_bonus_xgb_gain_top20", also_pdf=True)


In [ ]:
"""7.6 — KM by Diagnosis (cancer type) — clinical-context curve."""
if not KM_AVAILABLE or train is None:
    print("(skip)")
else:
    surv = _surv_df(train)
    dx_col = _first_present(train.obs, ["Diagnosis", "diagnosis", "CancerType", "cancer_type"])
    if surv is None or dx_col is None:
        print("Skip KM-diagnosis")
    else:
        df = surv.copy()
        df["Diagnosis"] = train.obs.loc[surv.index, dx_col].astype(str).fillna("NA").values
        df = df[df["Diagnosis"] != "NA"]
        if df["Diagnosis"].nunique() < 2:
            print("(skip)")
        else:
            fig, ax = plt.subplots(figsize=(10, 7))
            kmf = KaplanMeierFitter()
            levels = sorted(df["Diagnosis"].unique())
            pal = sns.color_palette("Set2", max(3, len(levels)))
            for i, lv in enumerate(levels):
                sub = df[df["Diagnosis"] == lv]
                if sub.shape[0] < 5:
                    continue
                kmf.fit(sub["time"], sub["event"], label=f"{lv} (n={len(sub)})")
                kmf.plot_survival_function(
                    ax=ax, ci_show=True, color=pal[i % len(pal)], linewidth=2.2,
                )
            res = multivariate_logrank_test(df["time"], df["Diagnosis"], df["event"])
            ax.set_title(
                f"Kaplan–Meier — stratified by Diagnosis  (log-rank p={res.p_value:.3g})"
            )
            ax.set_xlabel("Time (days)"); ax.set_ylabel("Survival probability")
            ax.set_ylim(0, 1.02)
            ax.legend(frameon=False)
            _save(fig, "sota_km_by_diagnosis", also_pdf=True)


In [ ]:
"""7.7 — UMAP coloured by predicted risk tertile (uses XGBoost + cAE if available)."""
if umap_xy is None or xgb_model is None or train is None:
    print("(skip)")
else:
    try:
        risk = xgb_model.predict_proba(X_shap)[:, 1]
    except Exception as exc:
        print(f"(skip: {exc})")
    else:
        # The xgb_model was trained on labelled rows only; risk has that shape.
        # Pad to full obs index so we can colour every UMAP point.
        full_risk = np.full(umap_xy.shape[0], np.nan, dtype=np.float32)
        labelled_idx = np.array([train.obs.index.get_loc(i) for i in X_shap.index])
        full_risk[labelled_idx] = risk
        tert = pd.qcut(
            pd.Series(full_risk).dropna(), 3,
            labels=["Low", "Mid", "High"], duplicates="drop",
        )
        full_tert = pd.Series(["NA"] * len(full_risk), index=range(len(full_risk)))
        full_tert.iloc[labelled_idx] = tert.astype(str).values
        fig, ax = plt.subplots(figsize=(8.5, 7))
        for lv, col in zip(["Low", "Mid", "High", "NA"],
                           ["#3182ce", "#ecc94b", "#e53e3e", "#cbd5e0"]):
            mask = full_tert.values == lv
            if mask.sum() == 0:
                continue
            ax.scatter(
                umap_xy[mask, 0], umap_xy[mask, 1],
                c=col, s=20, alpha=0.85, edgecolor="white", linewidth=0.4,
                label=f"{lv} (n={int(mask.sum())})",
            )
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
        ax.set_title("cAE latent space — predicted response-risk tertile (XGBoost p̂)")
        ax.legend(frameon=False)
        _save(fig, "sota_umap_predicted_risk_tertile")


In [ ]:
"""7.4 — Confusion matrices grid for the best classifier per embedding (TRAIN, OOF-style)."""
from sklearn.model_selection import StratifiedKFold  # noqa: E402
from sklearn.metrics import confusion_matrix  # noqa: E402

if train is None:
    print("(skip)")
else:
    embs = [(k, lbl) for k, lbl in [
        (CAE_KEY,      "cAE"),
        (SCGPT_KEY,    "Raw scGPT"),
        (SCGPT_FT_KEY, "scGPT-FT"),
    ] if k in train.obsm]
    resp_col = _first_present(train.obs, [
        "Response", "response", "Responder", "responder", "RECIST", "recist",
    ])
    if not embs or resp_col is None:
        print("(skip: no labelled response or no embeddings)")
    else:
        pos = {"r", "cr", "pr", "responder", "yes", "1", "true"}
        neg = {"nr", "sd", "pd", "non_responder", "no", "0", "false"}
        raw = train.obs[resp_col].astype(str).str.lower().str.strip()
        y = raw.map(lambda v: 1 if v in pos else (0 if v in neg else np.nan))
        mask = ~y.isna()
        y = y[mask].astype(int).to_numpy()
        fig, axes = plt.subplots(1, len(embs), figsize=(4.6 * len(embs), 4.6))
        if len(embs) == 1:
            axes = [axes]
        for ax, (key, lbl) in zip(axes, embs):
            X = np.asarray(train.obsm[key], dtype=np.float32)[mask.values]
            scl = _SS().fit(X)
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            preds = np.zeros_like(y)
            for tr_i, va_i in skf.split(X, y):
                clf = LogisticRegression(max_iter=2000, C=0.5).fit(
                    scl.transform(X[tr_i]), y[tr_i]
                )
                preds[va_i] = clf.predict(scl.transform(X[va_i]))
            cm = confusion_matrix(y, preds)
            sns.heatmap(
                cm, annot=True, fmt="d", cmap="Blues",
                cbar=False, ax=ax, square=True,
                xticklabels=["NR", "R"], yticklabels=["NR", "R"],
            )
            acc = (preds == y).mean()
            ax.set_title(f"{lbl}\nacc={acc:.3f}")
            ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        fig.suptitle(
            "OOF Confusion matrices — LogReg on each embedding",
            y=1.04, fontsize=13, weight="bold",
        )
        _save(fig, "sota_bonus_confusion_grid", also_pdf=True)


## ✓ Output index

All PNGs (and most PDFs) live in `visualizations/sota_*.png`. Listing below
makes it easy to embed them in the final report.

In [ ]:
produced = sorted(VIZ_DIR.glob("sota_*.png"))
print(f"Total SOTA plots produced: {len(produced)}\n")
for p in produced:
    print(f"  - {p.relative_to(REPO)}")
